### Important links

- https://docs.databricks.com/aws/en/admin/users-groups/manage-service-principals?language=Workspace%C2%A0level%C2%A0deactivation

- https://docs.databricks.com/aws/en/dev-tools/cli/reference/secrets-commands




#### External location (aws/azure later)

#### Key Security Practices

- Permissions: Use the Secrets CLI or UI to ensure only authorized users/groups have READ access to the secret scope.
- Redaction: Databricks automatically redacts values retrieved via dbutils.secrets.get() in notebook output to prevent leaking the secret.


#### Create the Secret Scope

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import SecretScopePermission

w = WorkspaceClient()
w.secrets.create_scope(scope="<scope-name>")
w.secrets.put_acl(
    scope="<scope-name>",
    principal="<application-id>",
    permission=SecretScopePermission.READ
)

#### AWS

In [0]:
# Define your scope and keys / scope_name is coming from key-vault / access_key_key and secret_key_key are coming secret scope created in databricks.
scope_name = "key-vault-secrets" 
access_key_key = "aws-access-key"
secret_key_key = "aws-secret-key"

# Retrieve credentials securely using dbutils
aws_access_key = dbutils.secrets.get(scope=scope_name, key=access_key_key)
aws_secret_key = dbutils.secrets.get(scope=scope_name, key=secret_key_key)

# Configure Spark to use AWS credentials for S3 access
spark.conf.set("fs.s3a.access.key", aws_access_key)
spark.conf.set("fs.s3a.secret.key", aws_secret_key)

# Read data from the external source
bucket_name = "mybucket"
path = "path/to/data"
df = spark.read.format("parquet").load(f"s3a://{bucket_name}/{path}")
display(df)


#### Azure

In [0]:
# Define your scope and keys / scope_name is coming from key-vault / client_id_key and client_secret_key are coming secret scope created in databricks.
scope_name = "key-vault-secrets" 
client_id_key = "sp-client-id"
client_secret_key = "sp-client-secret" 
tenant_id = "your-tenant-id-uuid"

# Retrieve credentials securely using dbutils
client_id = dbutils.secrets.get(scope=scope_name, key=client_id_key)
client_secret = dbutils.secrets.get(scope=scope_name, key=client_secret_key)

# Configure Spark to use the Service Principal for ADLS Gen2
storage_account = "mystorageaccount"
container_name = "rawdata"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com{tenant_id}/oauth2/token")

# Read data from the external source
df = spark.read.format("parquet").load(f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/path/to/data")
display(df)
